In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch 
import numpy as np
import random
torch.autograd.set_detect_anomaly(True)
torch.multiprocessing.set_sharing_strategy("file_descriptor")
seed = 140421
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [ ]:
from detectron2.data.datasets.pano360 import CalibDataset

debug = True
train_calib = CalibDataset(
    train=True,
    json_name="datasets/pano360_crops_dataset_cvpr_myDistWider_train.json",
    logger=None,
    debug=debug,
)
val_calib = CalibDataset(
    train=False,
    json_name="datasets/pano360_crops_dataset_cvpr_myDistWider_train.json",
    logger=None,
    debug=debug,
)

In [ ]:
import cv2
import matplotlib.pyplot as plt
from detectron2.data.datasets.pano360 import bins2pitch, bins2roll, bins2vfov, bins2horizon, showHorizonLineFromHorizon, showHorizonLine
from detectron2.utils.visualizer import Visualizer

if debug and len(train_calib) < 500:
    dataset_dicts = list(train_calib.get_all_items())
    for i, d in enumerate(random.sample(dataset_dicts, 3)):
        print(d.keys())
        img = cv2.imread(d["file_name"])
        visualizer = Visualizer(img[:, :, ::-1], scale=0.5)
        out = visualizer.get_output()
        img = out.get_image()
        # plt.imshow(img)
        gt_horizon = bins2horizon(d["logits"]["gt_horizon"])
        gt_pitch = bins2pitch(d["logits"]["gt_pitch"])
        gt_roll = bins2roll(d["logits"]["gt_roll"])
        gt_vfov = bins2vfov(d["logits"]["gt_vfov"])
        anno_img, _ = showHorizonLine(img, gt_vfov, gt_pitch, gt_roll)
        anno_img = showHorizonLineFromHorizon(anno_img, gt_horizon, color=(255, 255, 255), width=3, debug=True, GT=True, ) # White: GT horizon without roll
        plt.imshow(anno_img)
        plt.show()

In [ ]:
from detectron2.data import DatasetCatalog
DatasetCatalog.register("Pano360_train", train_calib)
DatasetCatalog.register("Pano360_val", val_calib)

In [ ]:
import os
from detectron2.engine import CalibTrainer
from detectron2 import model_zoo
from detectron2.config import get_cfg

cfg = get_cfg()
config_path = "COCO-Keypoints/keypoint_rcnn_R_50_FPN_3x.yaml"
cfg.merge_from_file(model_zoo.get_config_file(config_path))
# cfg.DATALOADER
cfg.DATASETS.TRAIN = ("Pano360_train",)
cfg.DATASETS.TEST = ("Pano360_val",)
cfg.DATALOADER.NUM_WORKERS = 4
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(config_path)  # Let training initialize from model zoo
cfg.SOLVER.IMS_PER_BATCH = 4  # This is the real "batch size" commonly known to deep learning people
cfg.SOLVER.BASE_LR = 0.00025  # pick a good LR
cfg.MODEL.META_ARCHITECTURE = "CameraRCNN"
cfg.MODEL.PROPOSAL_GENERATOR.NAME = "PrecomputedProposals"  # Disable RPN Network. -- Must add proposals to input data
cfg.MODEL.KEYPOINT_ON=False
cfg.VIS_PERIOD = 10
cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS=False
cfg.SOLVER.MAX_ITER = 500
experiment_name = "test-debug-calib-dataset"
cfg.OUTPUT_DIR = os.path.join("output", experiment_name)
cfg.SOLVER.AMP.ENABLED = True  # Enable AMP here -- improve 10s per iter approx.

trainer = CalibTrainer(cfg) 
# NOTE: change value of resume if we have a last_checkpoint
trainer.resume_or_load(resume=False)

In [ ]:
trainer.train()

In [ ]:
from detectron2.evaluation import Pano360Evaluator

trainer.model.eval()
trainer.test(cfg, trainer.model, evaluators=Pano360Evaluator())